In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from scipy.io.arff import loadarff


In [3]:
train_path="Data/ECG200_TRAIN.arff"
test_path="Data/ECG200_TEST.arff"
def read_ariff(path):
    raw_data, meta =loadarff(path)
    cols=[x for x in meta]
    data2d=np.zeros([raw_data.shape[0],len(cols)])
    for i,col in zip(range(len(cols)),cols):
        data2d[:,i]=raw_data[col]
    print(data2d.shape)
    return data2d

data2d=read_ariff(train_path)
test2d=read_ariff(test_path)
print(data2d.shape, test2d.shape)

(100, 97)
(100, 97)
(100, 97) (100, 97)


In [4]:
print(test2d)

[[ 0.42518938  1.4185988   2.6687913  ... -0.14248753  0.09875814
   1.        ]
 [ 0.65392929  2.1772899   3.6447828  ...  0.17110719  0.19702731
   1.        ]
 [ 0.4049532   0.55399598  0.72409707 ...  0.81257     0.39063926
   1.        ]
 ...
 [ 1.1136848   1.2759515   1.1717696  ...  0.47963844  0.35869955
  -1.        ]
 [ 2.3182077   2.1397723   1.7942428  ...  0.28500798  0.4709143
  -1.        ]
 [ 2.3953288   3.2836971   2.9185984  ...  0.07265965  0.21594169
   1.        ]]


import the required libraries

Reading the wafer dataset from the time series website. Download from https://www.timeseriesclassification.com/description.php?Dataset=Wafer

Dataset reading - preprocessing

In [5]:
# Get the number of columns
cols = data2d.shape[1]

# Split the data
train_x = data2d[:, :cols-1]  # All columns except the last one
train_y = data2d[:, cols-1]   # The last column



In [6]:
train_y

array([-1.,  1., -1., -1.,  1.,  1., -1., -1.,  1.,  1.,  1.,  1.,  1.,
        1., -1.,  1.,  1.,  1., -1., -1.,  1.,  1.,  1., -1.,  1.,  1.,
        1., -1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1., -1., -1.,  1., -1., -1.,  1., -1.,  1.,  1.,
        1., -1.,  1.,  1., -1., -1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,
       -1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,  1., -1.,  1., -1.,
        1.,  1., -1.,  1.,  1., -1., -1.,  1.,  1.])

In [ ]:
train_y[np.where(train_y != 1)] = 0

In [5]:
import numpy as np

def quantize(value, num_bits):
    """Quantizes a floating-point value to a fixed-point representation."""
    min_val = -2.0
    max_val = 2.0
    scale = (max_val - min_val) / (2 ** num_bits - 1)
    quantized_value = np.round(value / scale)
    return quantized_value * scale

# Example usage
x = np.array([0.995, -1.1481812, -1.1804904])
quantized_x = quantize(x, 10)
print(quantized_x)

[ 0.99315738 -1.14956012 -1.18084066]


In [6]:
import numpy as np

def quantize_to_nbit_signed(value, n, scale_factor=1000):
    """Quantizes a floating-point value to a 10-bit signed integer representation with scaling."""
    min_val = -1 * (2 ** (n - 1))
    max_val = (2 ** (n - 1))
    # Apply the scale factor to the value
    scaled_value = value * scale_factor
    # Calculate scale based on the range of the 10-bit signed integer
    scale = (max_val - min_val) / (2 ** n - 1)
    # Quantize the value to the nearest integer within the range
    quantized_value = np.clip(np.round(scaled_value / scale), min_val, max_val)
    return quantized_value

# Example usage
x = np.array([0.001, -1.1481812, -1.1804904])
quantized_x = quantize_to_nbit_signed(x, 11)
print(quantized_x)


[ 1.000e+00 -1.024e+03 -1.024e+03]


In [7]:
import numpy as np

def quantize_to_nbit_signed_int(value, num_bits, scale_factor=None):
    """Quantizes a floating-point value to an n-bit signed integer representation with dynamic scaling."""
    min_val = -2**(num_bits - 1)
    max_val = 2**(num_bits - 1) - 1

    # If scale_factor is not provided, calculate it based on the max absolute value in 'value'
    if scale_factor is None:
        # Find the maximum absolute value in the input array to map the range exactly
        max_abs_value = np.max(np.abs(value))
        # Compute scale factor to map max_abs_value to fit within the range of n-bit integers
        scale_factor = max_val / max_abs_value

    # Apply the calculated scale factor to scale the input value to the target range
    scaled_value = value * scale_factor
    # Calculate scale for quantization based on the integer range
    scale = (max_val - min_val) / (2 ** num_bits - 1)
    # Quantize the value to the nearest integer within the range and convert to int
    quantized_value = np.clip(np.round(scaled_value / scale), min_val, max_val).astype(int)
    return quantized_value, scale_factor  # Return scale_factor to see the scaling applied

# Example usage
x = np.array([0.995, 0.7, -0.2])
quantized_x, scale_factor = quantize_to_nbit_signed_int(x, num_bits=10)
print("Quantized Values:", quantized_x)
print("Scale Factor:", scale_factor)


Quantized Values: [ 511  359 -103]
Scale Factor: 513.5678391959799


In [8]:
import numpy as np
import math

def quantize_to_flexible_bits(value, scale_factor=10000, num_bits=None):
    """
    Quantizes a floating-point value to a flexible signed integer representation.
    If `num_bits` is not specified, the function determines the nearest power of 2 bits.
    """
    # Step 1: Scale the input values
    scaled_value = value * scale_factor
    # Step 2: Determine the maximum absolute value for bit-width calculations
    max_abs_value = np.max(np.abs(scaled_value))

    # Handle the case where max_abs_value is zero
    if max_abs_value == 0:
        max_range = 0
    else:
        # Step 3: Round up to the nearest power of 10
        max_range = 10 ** math.ceil(math.log10(max_abs_value))

    # Step 2: Get the maximum absolute value of the scaled input
    #max_abs_value = np.max(np.abs(scaled_value))

    # Step 3: Round up to the nearest power of 10
    #max_range = 10 ** math.ceil(math.log10(max_abs_value))
    # Step 4: Determine the bit width if not provided


    # Step 4: Determine the bit width if not provided
    if num_bits is None:
        # Calculate the minimum number of bits to cover the max_range
        if max_range != 0:
          num_bits = math.ceil(math.log2(max_range * 2))
        else:
          num_bits = 1



    # Calculate the integer min and max based on num_bits
    #min_val = -2**(num_bits - 1)
    #max_val = 2**(num_bits - 1) - 1
    min_val = -1 * max_range
    max_val = max_range


    # Calculate the scaling factor for quantization
    scale = (max_val - min_val) / (2 ** num_bits - 1)

    # Quantize the scaled value to the range and convert to int
    if max_range == 0:
      quantized_value = 0
    else:
      quantized_value = np.clip(np.round(scaled_value / scale), min_val, max_val).astype(int)

    return quantized_value, num_bits, max_range

# Example usage
x = np.array([0.995, 0.7, -0.2])
quantized_x, used_bits, max_range = quantize_to_flexible_bits(x,scale_factor=10000)
print("Quantized Values:", quantized_x)
print("Bits Used:", used_bits)
print("Max Range:", max_range)


Quantized Values: [10000 10000 -3277]
Bits Used: 15
Max Range: 10000


In [9]:
import numpy as np
import torch
import math

def quantize_scalar_to_flexible_bits(value, scale_factor=10000, num_bits=None):
    """
    Quantizes a scalar floating-point value to a flexible signed integer representation.
    If `num_bits` is not specified, the function determines the nearest power of 2 bits.

    Args:
        value (float or torch.Tensor): The scalar value to be quantized.
        scale_factor (int): Scale factor for quantization.
        num_bits (int, optional): Desired number of bits for quantization.

    Returns:
        tuple: A tuple containing the quantized integer (or tensor), bits used, and max range.
    """
    # Convert PyTorch tensor to NumPy for processing if needed
    tensor_input = isinstance(value, torch.Tensor)
    if tensor_input:
        value = value.item()  # Convert single-element tensor to a scalar

    # Step 1: Scale the input value
    scaled_value = value * scale_factor

    # Step 2: Determine the maximum absolute value for bit-width calculations
    max_abs_value = abs(scaled_value)

    # Handle the case where max_abs_value is zero
    if max_abs_value == 0:
        max_range = 0
    else:
        # Step 3: Round up to the nearest power of 10
        max_range = 10 ** math.ceil(math.log10(max_abs_value))

    # Step 4: Determine the bit width if not provided
    if num_bits is None:
        # Calculate the minimum number of bits to cover the max_range
        if max_range != 0:
            num_bits = math.ceil(math.log2(max_range * 2))
        else:
            num_bits = 1

    # Calculate the integer min and max based on num_bits
    min_val = -max_range
    max_val = max_range

    # Calculate the scaling factor for quantization
    scale = (max_val - min_val) / (2 ** num_bits - 1)

    # Quantize the scaled value to the range and convert to int
    if max_range == 0:
        quantized_value = 0
    else:
        quantized_value = int(np.clip(round(scaled_value / scale), min_val, max_val))

    # Convert back to PyTorch tensor if the input was a tensor
    if tensor_input:
        quantized_value = torch.tensor(quantized_value, dtype=torch.int32)

    return quantized_value, num_bits, max_range

# Example usage
value = torch.tensor(0.995)
quantized_value, used_bits, max_range = quantize_scalar_to_flexible_bits(value, scale_factor=10000)
print("Quantized Value:", quantized_value)
print("Bits Used:", used_bits)
print("Max Range:", max_range)


Quantized Value: tensor(10000, dtype=torch.int32)
Bits Used: 15
Max Range: 10000


BaseLDN

In [ ]:
import numpy as np

class BaseLDN:
    """
    A simplified Legendre Delay Network (LDN) using only NumPy.
    Implements a continuous-time delay of an input signal.
    """

    def __init__(self, order, theta, tau, dt=0.001, c2d=True):
        """
        Initialize the LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        self.order = order
        self.theta = theta
        self.tau = tau
        self.dt = dt
        self.c2d = c2d

        self._init_A_and_B_matrices()
        self._init_A_p_and_B_p_matrices()

    def _init_A_and_B_matrices(self):
        """
        Initialize continuous-time A and B matrices.
        """
        Q = np.arange(self.order)
        R = (2 * Q + 1)[:, None] / self.theta
        j, i = np.meshgrid(Q, Q)


        # Calculate continuous-time A and B matrices
        self.A = np.where(i < j, -1, (-1.0)**(i - j + 1)) * R
        self.B = (-1.0)**Q[:, None] * R
        print(self.A)
        print(self.B)

    def _init_A_p_and_B_p_matrices(self):
        """
        Initialize A' and B' matrices based on A, B, tau, and c2d.
        """
        if self.c2d:
            # Continuous-to-discrete using simple approximation
            self.A_p = np.eye(self.order) + self.A * self.dt
            self.B_p = self.B * self.dt
        else:
            # Continuous-time A' and B' matrices
            self.A_p = self.A * self.tau + np.eye(self.order)
            self.B_p = self.B * self.tau



    def get_transformed_matrices(self):
        """
        Returns the transformed (A' and B') matrices.
        """
        return self.A_p, self.B_p


In [ ]:
import numpy as np
from scipy.signal import cont2discrete

class BaseLDN:
    """
    A simplified Legendre Delay Network (LDN) using only NumPy.
    Implements a continuous-time delay of an input signal.
    """

    def __init__(self, order, theta, tau, dt=0.01, c2d=True):
        """
        Initialize the LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        self.order = order
        self.theta = theta
        self.tau = tau
        self.dt = dt
        self.c2d = c2d

        self._init_A_and_B_matrices()
        self._init_A_p_and_B_p_matrices()

    def _init_A_and_B_matrices(self):
        """
        Initialize continuous-time A and B matrices.
        """
        Q = np.arange(self.order)
        R = (2 * Q + 1)[:, None] / self.theta
        j, i = np.meshgrid(Q, Q)


        # Calculate continuous-time A and B matrices
        self.A = np.where(i < j, -1, (-1.0)**(i - j + 1)) * R
        self.B = (-1.0)**Q[:, None] * R


    def _init_A_p_and_B_p_matrices(self):
        """
        Initialize A' and B' matrices based on A, B, tau, and c2d.
        """
        if self.c2d:
            C = np.ones((1, self.order))
            D = np.zeros((1,))
            self.A_p, self.B_p, _, _, _ = cont2discrete((self.A, self.B, C, D), dt=self.dt, method="zoh")
        else:
            # Continuous-time A' and B' matrices
            self.A_p = self.A * self.tau + np.eye(self.order)
            self.B_p = self.B * self.tau



    def get_transformed_matrices(self):
        """
        Returns the transformed (A' and B') matrices.
        """
        return self.A_p, self.B_p


In [ ]:
import numpy as np

# Given parameters
order = 8
theta = 1.0
tau = 0.1
c2d = True  # Ensure continuous-time transformation

# Create BaseLDN instance
ldn = BaseLDN(order=order, theta=theta, tau=tau, c2d=c2d)

# Get transformed matrices
A_p, B_p = ldn.get_transformed_matrices()

# Print results
np.set_printoptions(precision=4, suppress=True)  # Set formatting for readability
print("A' matrix:")
print(A_p)
print("\nB' matrix:")
print(B_p)


[[ -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  3.  -3.  -3.  -3.  -3.  -3.  -3.  -3.]
 [ -5.   5.  -5.  -5.  -5.  -5.  -5.  -5.]
 [  7.  -7.   7.  -7.  -7.  -7.  -7.  -7.]
 [ -9.   9.  -9.   9.  -9.  -9.  -9.  -9.]
 [ 11. -11.  11. -11.  11. -11. -11. -11.]
 [-13.  13. -13.  13. -13.  13. -13. -13.]
 [ 15. -15.  15. -15.  15. -15.  15. -15.]]
[[  1.]
 [ -3.]
 [  5.]
 [ -7.]
 [  9.]
 [-11.]
 [ 13.]
 [-15.]]
A' matrix:
[[ 0.999 -0.001 -0.001 -0.001 -0.001 -0.001 -0.001 -0.001]
 [ 0.003  0.997 -0.003 -0.003 -0.003 -0.003 -0.003 -0.003]
 [-0.005  0.005  0.995 -0.005 -0.005 -0.005 -0.005 -0.005]
 [ 0.007 -0.007  0.007  0.993 -0.007 -0.007 -0.007 -0.007]
 [-0.009  0.009 -0.009  0.009  0.991 -0.009 -0.009 -0.009]
 [ 0.011 -0.011  0.011 -0.011  0.011  0.989 -0.011 -0.011]
 [-0.013  0.013 -0.013  0.013 -0.013  0.013  0.987 -0.013]
 [ 0.015 -0.015  0.015 -0.015  0.015 -0.015  0.015  0.985]]

B' matrix:
[[ 0.001]
 [-0.003]
 [ 0.005]
 [-0.007]
 [ 0.009]
 [-0.011]
 [ 0.013]
 [-0.015]]


In [ ]:
import numpy as np
from scipy.linalg import expm

class BaseLDN:
    """
    A simplified Legendre Delay Network (LDN) using only NumPy.
    Implements a continuous-time delay of an input signal.
    """

    def __init__(self, order, theta, tau, dt=0.001, c2d=True):
        """
        Initialize the LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        self.order = order
        self.theta = theta
        self.tau = tau
        self.dt = dt
        self.c2d = c2d

        self._init_A_and_B_matrices()
        self._init_A_p_and_B_p_matrices()

    def _init_A_and_B_matrices(self):
        """
        Initialize continuous-time A and B matrices.
        """
        Q = np.arange(self.order)
        R = (2 * Q + 1)[:, None] / self.theta
        j, i = np.meshgrid(Q, Q)


        # Calculate continuous-time A and B matrices
        self.A = np.where(i < j, -1, (-1.0)**(i - j + 1)) * R
        self.B = (-1.0)**Q[:, None] * R


    def _init_A_p_and_B_p_matrices(self):
        """
        Initialize A' and B' matrices based on A, B, tau, and c2d.
        """
        if self.c2d:
             self.A_p = expm(self.A * self.dt)

        # Compute B_p using the formula B_p = A^(-1) (e^(A dt) - I) B
             self.B_p = np.linalg.solve(self.A, (self.A_p - np.eye(self.order)) @ self.B)
        else:
            # Continuous-time A' and B' matrices
            self.A_p = self.A * self.tau + np.eye(self.order)
            self.B_p = self.B * self.tau



    def get_transformed_matrices(self):
        """
        Returns the transformed (A' and B') matrices.
        """
        return self.A_p, self.B_p


In [ ]:
import numpy as np

# Given parameters
order = 8
theta = 1.0
tau = 0.1
c2d = True  # Ensure continuous-time transformation

# Create BaseLDN instance
ldn = BaseLDN(order=order, theta=theta, tau=tau, c2d=c2d)

# Get transformed matrices
A_p, B_p = ldn.get_transformed_matrices()

# Print results
np.set_printoptions(precision=4, suppress=True)  # Set formatting for readability
print("A' matrix:")
print(A_p)
print("\nB' matrix:")
print(B_p)


A' matrix:
[[ 0.999  -0.001  -0.001  -0.001  -0.001  -0.001  -0.001  -0.001 ]
 [ 0.003   0.997  -0.003  -0.003  -0.003  -0.0029 -0.003  -0.0029]
 [-0.005   0.005   0.995  -0.005  -0.005  -0.0049 -0.0049 -0.0049]
 [ 0.0069 -0.0069  0.007   0.993  -0.007  -0.0069 -0.0069 -0.0068]
 [-0.0089  0.009  -0.009   0.009   0.991  -0.0089 -0.0089 -0.0088]
 [ 0.0108 -0.0108  0.0108 -0.0109  0.0109  0.989  -0.011  -0.0108]
 [-0.0128  0.0128 -0.0128  0.0129 -0.0129  0.013   0.9869 -0.0129]
 [ 0.0145 -0.0145  0.0146 -0.0146  0.0147 -0.0147  0.0148  0.9851]]

B' matrix:
[[ 0.001 ]
 [-0.003 ]
 [ 0.005 ]
 [-0.0069]
 [ 0.0089]
 [-0.0108]
 [ 0.0128]
 [-0.0145]]


In [ ]:
import numpy as np

# Given parameters
order = 8
theta = 1.0
tau = 0.1
c2d = True  # Ensure continuous-time transformation

# Create BaseLDN instance
ldn = BaseLDN(order=order, theta=theta, tau=tau, c2d=c2d)

# Get transformed matrices
A_p, B_p = ldn.get_transformed_matrices()

# Print results
np.set_printoptions(precision=4, suppress=True)  # Set formatting for readability
print("A' matrix:")
print(A_p)
print("\nB' matrix:")
print(B_p)


A' matrix:
[[ 0.999  -0.001  -0.001  -0.001  -0.001  -0.001  -0.001  -0.001 ]
 [ 0.003   0.997  -0.003  -0.003  -0.003  -0.0029 -0.003  -0.0029]
 [-0.005   0.005   0.995  -0.005  -0.005  -0.0049 -0.0049 -0.0049]
 [ 0.0069 -0.0069  0.007   0.993  -0.007  -0.0069 -0.0069 -0.0068]
 [-0.0089  0.009  -0.009   0.009   0.991  -0.0089 -0.0089 -0.0088]
 [ 0.0108 -0.0108  0.0108 -0.0109  0.0109  0.989  -0.011  -0.0108]
 [-0.0128  0.0128 -0.0128  0.0129 -0.0129  0.013   0.9869 -0.0129]
 [ 0.0145 -0.0145  0.0146 -0.0146  0.0147 -0.0147  0.0148  0.9851]]

B' matrix:
[[ 0.001 ]
 [-0.003 ]
 [ 0.005 ]
 [-0.0069]
 [ 0.0089]
 [-0.0108]
 [ 0.0128]
 [-0.0145]]


In [ ]:
train_x.shape[1]

96

In [ ]:
train_y

array([0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
       1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
       1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1.,
       1., 1., 0., 1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.,
       1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
       0., 0., 1., 0., 1., 0., 1., 1., 0., 1., 1., 0., 0., 1., 1.])

In [ ]:

# Given parameters
order = 8
v_threshold = 1
bias = 0
gain = 1.0
num_neurons = 2 * order  # 16 neurons
timesteps = train_x.shape[1]  # Assuming there are 152 timesteps
#theta = 0.5
theta = 1.0
tau = 0.1

In [7]:
import numpy as np
import math

def quantize_matrix_to_flexible_bits(matrix, scale_factor=2**10, num_bits=None):
    """
    Quantizes a floating-point matrix to a flexible signed integer representation.
    If `num_bits` is not specified, the function determines the nearest power of 2 bits.
    """
    # Step 1: Scale the input matrix
    scaled_matrix = matrix * scale_factor

    # Step 2: Get the maximum absolute value of the scaled matrix
    max_abs_value = np.max(np.abs(scaled_matrix))
    #print("max_abs_value :", max_abs_value)

    # Step 3: Round up to the nearest power of 10
    max_range = 2 ** math.ceil(math.log2(max_abs_value))

    # Step 4: Determine the bit width if not provided
    if num_bits is None:
        # Calculate the minimum number of bits to cover the max_range
        num_bits = math.ceil(math.log2(max_range * 2))  # *2 to include both positive and negative range

    # Calculate the integer min and max based on num_bits
    #min_val = -2**(num_bits - 1)
    #max_val = 2**(num_bits - 1) - 1
    min_val = -1 * max_range
    max_val = max_range

    # Calculate the scaling factor for quantization
    scale = (max_val - min_val) / (2 ** num_bits - 1)

    # Quantize the scaled matrix element-wise to the range and convert to int
    quantized_matrix = np.clip(np.round(scaled_matrix / scale), min_val, max_val).astype(int)

    return quantized_matrix, num_bits, max_range

# Example usage
x = np.array([[0.995, 0.7, -0.2], [0.5, -0.6, 0.3]])
quantized_matrix, used_bits, max_range = quantize_matrix_to_flexible_bits(x)
print("Quantized Matrix:\n", quantized_matrix)
print("Bits Used:", used_bits)
print("Max Range:", max_range)


Quantized Matrix:
 [[1018  716 -205]
 [ 512 -614  307]]
Bits Used: 11
Max Range: 1024


In [8]:
x = train_x
quantized_matrix, used_bits, max_range = quantize_matrix_to_flexible_bits(x, scale_factor=2**10)
print("Quantized Matrix:\n", quantized_matrix)
print("Bits Used:", used_bits)
print("Max Range:", max_range)

Quantized Matrix:
 [[  514   555   740 ...   722   731   444]
 [  151   824   377 ...    38 -1295  -213]
 [  324   249   379 ...   767   838   552]
 ...
 [  202   466   996 ...  -123    43   352]
 [  184  1063  1993 ...   155   198   463]
 [   75   795  2234 ...   -68  -183  -262]]
Bits Used: 14
Max Range: 8192


In [9]:
np.savetxt("quantized_input_train_ECG200.csv", quantized_matrix, delimiter=",", fmt="%d")

In [29]:
import numpy as np
from scipy.linalg import expm

class BaseLDN:
    """
    A simplified Legendre Delay Network (LDN) using only NumPy.
    Implements a continuous-time delay of an input signal.
    """

    def __init__(self, order, theta, tau, dt=0.001, c2d=True):
        """
        Initialize the LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        self.order = order
        self.theta = theta
        self.tau = tau
        self.dt = dt
        self.c2d = c2d

        self._init_A_and_B_matrices()
        self._init_A_p_and_B_p_matrices()

    def _init_A_and_B_matrices(self):
        """
        Initialize continuous-time A and B matrices.
        """
        Q = np.arange(self.order)
        R = (2 * Q + 1)[:, None] / self.theta
        j, i = np.meshgrid(Q, Q)


        # Calculate continuous-time A and B matrices
        self.A = np.where(i < j, -1, (-1.0)**(i - j + 1)) * R
        self.B = (-1.0)**Q[:, None] * R


    def _init_A_p_and_B_p_matrices(self):
        """
        Initialize A' and B' matrices based on A, B, tau, and c2d.
        """
        if self.c2d:
             self.A_p = expm(self.A * self.dt)

        # Compute B_p using the formula B_p = A^(-1) (e^(A dt) - I) B
             self.B_p = np.linalg.solve(self.A, (self.A_p - np.eye(self.order)) @ self.B)
        else:
            # Continuous-time A' and B' matrices
            self.A_p = self.A * self.tau + np.eye(self.order)
            self.B_p = self.B * self.tau



    def get_transformed_matrices(self):
        """
        Returns the transformed (A' and B') matrices.
        """
        return self.A_p, self.B_p


In [31]:
# Define parameters
order = 8     # Order of the system
theta = 0.5   # Time constant for delay
tau = 0.1     # Low-pass filter time constant
dt = 0.1    # Time step for discretization

# Instantiate the BaseLDN class with continuous-to-discrete transformation
ldn_discrete = BaseLDN(order=order, theta=theta, tau=tau, dt=dt, c2d=True)

# Get and print the discrete-time A' and B' matrices
A_p_disc, B_p_disc = ldn_discrete.get_transformed_matrices()
np.set_printoptions(precision=4, suppress=True)  # Set formatting for readability

print("Discrete-time A':\n", A_p_disc)
print("Discrete-time B':\n", B_p_disc)


Discrete-time A':
 [[ 0.797  -0.1569 -0.0992 -0.0287  0.0084  0.0287  0.0179 -0.002 ]
 [ 0.4708  0.4254 -0.3746 -0.1322  0.006   0.0836  0.0607 -0.0021]
 [-0.496   0.6243  0.0616 -0.4153 -0.078   0.1188  0.1223  0.0109]
 [ 0.2009 -0.3085  0.5815 -0.1689 -0.4014  0.0676  0.1998  0.0472]
 [ 0.0754 -0.018  -0.1404  0.5161 -0.3768 -0.2781  0.2343  0.1111]
 [-0.316   0.3065 -0.2613  0.1062  0.3399 -0.505  -0.0154  0.1645]
 [ 0.2324 -0.2632  0.3181 -0.3711  0.3385  0.0182 -0.3311  0.0046]
 [ 0.0302 -0.0103 -0.0326  0.1011 -0.1852  0.2244 -0.0053 -0.1113]]
Discrete-time B':
 [[ 0.203 ]
 [-0.4708]
 [ 0.496 ]
 [-0.2009]
 [-0.0754]
 [ 0.316 ]
 [-0.2324]
 [-0.0302]]


In [12]:
#ECG200
import numpy as np
#A = np.array(A_p_disc)
A = np.array([
    [ 0.797 , -0.1569, -0.0992, -0.0287,  0.0084,  0.0287,  0.0179, -0.002 ],
    [ 0.4708,  0.4254, -0.3746, -0.1322,  0.006 ,  0.0836,  0.0607, -0.0021],
    [-0.496 ,  0.6243,  0.0616, -0.4153, -0.078 ,  0.1188,  0.1223,  0.0109],
    [ 0.2009, -0.3085,  0.5815, -0.1689, -0.4014,  0.0676,  0.1998,  0.0472],
    [ 0.0754, -0.018 , -0.1404,  0.5161, -0.3768, -0.2781,  0.2343,  0.1111],
    [-0.316 ,  0.3065, -0.2613,  0.1062,  0.3399, -0.505 , -0.0154,  0.1645],
    [ 0.2324, -0.2632,  0.3181, -0.3711,  0.3385,  0.0182, -0.3311,  0.0046],
    [ 0.0302, -0.0103, -0.0326,  0.1011, -0.1852,  0.2244, -0.0053, -0.1113]
])
#B = np.array(B_p_disc)
B = np.array([
    [ 0.203 ],
    [-0.4708],
    [ 0.496 ],
    [-0.2009],
    [-0.0754],
    [ 0.316 ],
    [-0.2324],
    [-0.0302]
])

AB_combined = np.hstack((B, A))

# Scale factor (2^16 = 65536)
scale_factor = 2**13

# Convert to integers by scaling
int_matrix = np.round(AB_combined * scale_factor).astype(int)
#int_matrix = AB_combined * scale_factor

print(int_matrix)
# Convert to 18-bit signed binary representation
def to_18bit_signed_bin(num):
    """Convert a signed integer to 18-bit binary format."""
    if num < 0:
        num = (1 << 18) + num  # Convert negative numbers using two's complement
    return format(num, '018b')

# Convert each element and print
binary_matrix = np.vectorize(to_18bit_signed_bin)(int_matrix)

# Print formatted output
for i, row in enumerate(binary_matrix):
    formatted_row = ", ".join([f"18'b{num}" for num in row])
    print(f"B_static[{i}] = {{{formatted_row}}}; // Row {i}")


[[ 1663  6529 -1285  -813  -235    69   235   147   -16]
 [-3857  3857  3485 -3069 -1083    49   685   497   -17]
 [ 4063 -4063  5114   505 -3402  -639   973  1002    89]
 [-1646  1646 -2527  4764 -1384 -3288   554  1637   387]
 [ -618   618  -147 -1150  4228 -3087 -2278  1919   910]
 [ 2589 -2589  2511 -2141   870  2784 -4137  -126  1348]
 [-1904  1904 -2156  2606 -3040  2773   149 -2712    38]
 [ -247   247   -84  -267   828 -1517  1838   -43  -912]]
B_static[0] = {18'b000000011001111111, 18'b000001100110000001, 18'b111111101011111011, 18'b111111110011010011, 18'b111111111100010101, 18'b000000000001000101, 18'b000000000011101011, 18'b000000000010010011, 18'b111111111111110000}; // Row 0
B_static[1] = {18'b111111000011101111, 18'b000000111100010001, 18'b000000110110011101, 18'b111111010000000011, 18'b111111101111000101, 18'b000000000000110001, 18'b000000001010101101, 18'b000000000111110001, 18'b111111111111101111}; // Row 1
B_static[2] = {18'b000000111111011111, 18'b111111000000100001

pytorch LDN

In [ ]:
import torch
import numpy as np

class PyTorchLDN(BaseLDN):
    def __init__(self, order, theta, tau, dt=0.001, c2d=True):
        """
        Initialize the PyTorch-based LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        # Initialize the BaseLDN (with A and B matrices)
        super().__init__(order, theta, tau, dt, c2d)

        # Get transformed A' and B' matrices from BaseLDN
        self.A_p, self.B_p = self.get_transformed_matrices()
        #print(self.A_p)
        #print(self.B_p)
        # Quantize A' and B' matrices and convert them to integers with a consistent dtype
        quantized_A_p, used_bits_A, max_range_A = quantize_matrix_to_flexible_bits(self.A_p, scale_factor=1000)
        quantized_B_p, used_bits_B, max_range_B = quantize_matrix_to_flexible_bits(self.B_p, scale_factor=1000)

        # Convert quantized matrices to integer Torch tensors with int32 dtype
        self.A_p = torch.from_numpy(quantized_A_p.astype(np.int32))
        self.B_p = torch.from_numpy(quantized_B_p.astype(np.int32))

        #print(self.A_p)
        #print(self.B_p)


    def get_ldn_transform(self, u_t, m):
        """
        Perform the LDN transformation with integer-only operations.

        Args:
            u_t <torch.Tensor>: Input scalar (1x1 tensor).
            m <torch.Tensor>: State-vector tensor.
        """
        # Quantize u_t before matrix multiplication and convert to integer with int32 dtype

        #quantized_u_t, used_bits_u, max_range_u = quantize_scalar_to_flexible_bits(u_t, scale_factor=1)
        #u_t = torch.from_numpy(np.array([[quantized_u_t]])).int()
        #print(u_t)
        #print("used_bits_u:\n", used_bits_u)
        # Ensure m is also of int32 dtype
        m = m.to(dtype=torch.int32)

        # Perform integer-only matrix multiplication
        return torch.mm(self.A_p, m) + torch.mm(self.B_p, u_t)

    def get_ldn_sigs(self, u):
        """
        Compute LDN signals using integer-only matrix multiplications.

        Args:
            u <torch.Tensor>: The input tensor (1xN).
        """
        if isinstance(u, np.ndarray):
            u = torch.from_numpy(u)

        # Ensure input tensor is integer with int32 dtype
        u = u.to(dtype=torch.int32)

        m = torch.zeros(self.order, 1, dtype=torch.int64)
        out = torch.zeros(u.shape[1], self.order, dtype=torch.int64)

        for i, u_t in enumerate(u[0]):
            u_t = u_t.reshape(1, 1)
            out[i, :] = self.get_ldn_transform(u_t, m).reshape(-1)
            m[:, 0] = out[i, :]  # Update the state vector

        #n = 0
        out = out.float() / 1000
        #out = out.float()
        return out


BASIC SPiking neuron

Encoder layer

In [ ]:
import torch
from abc import ABC, abstractmethod

class BaseSpikingNeuron(ABC):
    def __init__(self, v_threshold=1, gain=1, bias=0, dtype=torch.float32):
        self._dtype = dtype  # Set the data type manually
        self.v = torch.tensor(0.0, dtype=dtype)  # Membrane potential
        self.v_threshold = torch.tensor(v_threshold, dtype=dtype)
        self.gain = torch.tensor(gain, dtype=dtype)
        self.bias = torch.tensor(bias, dtype=dtype)

    def spike_and_reset(self):
        """
        Checks if the neuron spikes (membrane potential >= threshold) and resets voltage.
        Returns:
          spike (Tensor): A boolean tensor indicating the spike occurrence.
        """
        spike = self.v >= self.v_threshold  # Create boolean tensor for spike
        if torch.any(spike):  # If any spikes occurred
            self.v[spike] = 0.0  # Reset membrane potential for spiking neurons
        return spike.int()

    def reset_v(self):
        """Resets membrane potential to zero."""
        self.v = torch.tensor(0.0, dtype=self._dtype)

    @abstractmethod
    def encode(self, x):
        """Encodes input x. Needs to be implemented in derived class."""
        pass


In [ ]:
class Encoder(BaseSpikingNeuron):
    def __init__(self, encoder=1, v_threshold=1, gain=1, bias=0.5, dtype=torch.float32):
        super().__init__(v_threshold=v_threshold, gain=gain, bias=bias, dtype=dtype)
        self.encoder_value = torch.as_tensor(encoder, dtype=self._dtype)
        self.v = None  # Initialize v as None, to be created dynamically based on input size.

    def encode(self, x):
        """Encodes the input vector `x` into spikes using IF spiking mechanism."""
        x = torch.as_tensor(x, dtype=self._dtype)

        # Dynamically initialize self.v based on input size, if not already initialized
        if self.v is None or self.v.shape != x.shape:
            self.v = torch.zeros_like(x, dtype=self._dtype)

        # Ensure self.gain, self.encoder_value, and self.bias have shapes compatible with x
        if self.gain.dim() == 0:
            self.gain = self.gain.expand_as(x)
        if self.encoder_value.dim() == 0:
            self.encoder_value = self.encoder_value.expand_as(x)
        if self.bias.dim() == 0:
            self.bias = self.bias.expand_as(x)

        # Compute the input current (J) based on gain, encoder, and bias.
        current_input = self.gain * self.encoder_value * x + self.bias

        # Update the membrane potential (v) by adding current.
        self.v += current_input

        # Ensure membrane potential does not go negative (rectification).
        self.v[self.v < 0] = 0.0

        # Check for spikes and reset membrane potential where spike occurs.
        spike = self.spike_and_reset()

        return spike


In [ ]:


class EncoderLayer(Encoder):
    def __init__(self, order, num_neurons, v_threshold=1, gain=1, bias=0.5, dtype=torch.float32):
        super().__init__(v_threshold=v_threshold, gain=gain, bias=bias, dtype=dtype)
        self.order = order
        self.num_neurons = num_neurons  # Should be 2 * order (for duplication)

    def duplicate_input(self, x):
        """Duplicates each element in the input tensor `x` to create a new tensor of size 2d."""
        x_dup = torch.repeat_interleave(x, 2)  # Duplicate each element in x
        return x_dup

    def create_alternating_encoder_values(self, length):
        """Creates alternating encoder values [1, -1, 1, -1, ...] of size `length`."""
        encoder_values = torch.ones(length, dtype=self._dtype)
        encoder_values[1::2] = -1  # Set every second element to -1
        return encoder_values

    def encode_layer(self, x):
        """Encodes the duplicated input `x` using the Encoder mechanism with alternating encoder values."""
        # Duplicate the input tensor (convert order-length tensor to 2*order-length tensor)
        x_dup = self.duplicate_input(x)

        # Set encoder values to alternate between 1 and -1 for 2*order length
        self.encoder_value = self.create_alternating_encoder_values(x_dup.shape[0])

        # Use the inherited encode function to process the duplicated input
        spikes = self.encode(x_dup)

        return spikes

In [ ]:
import torch

# Given parameters
order = 12
v_threshold = 1
bias = 0
gain = 1.0
num_neurons = 2 * order  # 16 neurons
timesteps = train_x.shape[1]  # Assuming there are 152 timesteps
#theta = 0.5
theta = 1.0
tau = 0.1
# Create an instance of EncoderLayer
encoder = EncoderLayer(order=order, num_neurons=num_neurons, v_threshold=v_threshold, gain=gain, bias=bias)
#train_x = test_x
#train_y = test_y
# Initialize a list to hold the stacked spikes for all samples
stacked_spikes = []
ldn_all_signals = []
# Process each sample in train_x
for i in range(train_x.shape[0]):  # Loop over all samples
    first_sample = train_x[i, :].reshape(1, -1)  # Reshape to match input format

    # Create an instance of PyTorchLDN for each sample
    ldn = PyTorchLDN(order, theta, tau)

    # Compute the LDN signals for the current sample
    ldn_first_sample_signals = ldn.get_ldn_sigs(first_sample)
    #print(ldn_first_sample_signals)

    # Initialize a list to hold spikes for the current sample
    current_sample_spikes = []
    ldn_all_signals.append(ldn_first_sample_signals)

    # Encoding over 152 timesteps using LDN output as input
    for t in range(timesteps):
        x_t = ldn_first_sample_signals[t]  # Extract the input for timestep `t` (size `order`)
        spikes = encoder.encode_layer(x_t)  # Process input and generate spikes
        current_sample_spikes.append(spikes)  # Collect spikes for current sample
        #print(spikes)
    # Stack the spikes for the current sample
    current_sample_spikes_tensor = torch.stack(current_sample_spikes)  # Shape will be [152, 16]
    if i==0:
      first_sample_xt = ldn_first_sample_signals
      first_sample_spikes = current_sample_spikes_tensor


    # Transpose to get shape [16, 152]
    combined_output = current_sample_spikes_tensor.transpose(0, 1)  # Shape will be [16, 152]

    # Append the combined output for the current sample
    stacked_spikes.append(combined_output)

# Stack all samples into a single tensor
final_output = torch.stack(stacked_spikes)  # Shape will be [6164, 16, 152]

# Print the final output shape
print("Final output shape:", final_output.shape)  # Should be [6164, 16, 152]


Final output shape: torch.Size([100, 24, 16])


In [ ]:
first_sample_xt[0] * 1024

tensor([0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
first_sample_xt[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
first_sample_spikes[0]


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=torch.int32)

In [ ]:
first_sample_spikes[1]

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=torch.int32)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Sample dimensions
batch_size = 1000
d = order
n = train_x.shape[1]
input_dim = 2 * d * n  # Flattened input size (16 * 152)


#train_x = torch.randn(batch_size, 2 * d, n)  # Shape: (999, 16, 152)
train_x = final_output.float()
#train_y = torch.randint(0, 2, (batch_size,))  # Binary labels for two classes
# Convert train_y from numpy array to torch tensor
train_y_tensor = torch.tensor(train_y, dtype=torch.long)  # Ensure the correct dtype for classification

# Define the model
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(input_dim, 2)  # Fully connected layer for 2 classes

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten 2*d x n to (batch_size, 2*d*n)
        return self.fc(x)

# Instantiate the model
model = Classifier(input_dim)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 40
for epoch in range(num_epochs):
    model.train()

    # Forward pass
    outputs = model(train_x)
    loss = criterion(outputs, train_y_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [1/40], Loss: 0.7451
Epoch [2/40], Loss: 0.6942
Epoch [3/40], Loss: 0.6594
Epoch [4/40], Loss: 0.6383
Epoch [5/40], Loss: 0.6266
Epoch [6/40], Loss: 0.6196
Epoch [7/40], Loss: 0.6136
Epoch [8/40], Loss: 0.6066
Epoch [9/40], Loss: 0.5979
Epoch [10/40], Loss: 0.5877
Epoch [11/40], Loss: 0.5766
Epoch [12/40], Loss: 0.5655
Epoch [13/40], Loss: 0.5551
Epoch [14/40], Loss: 0.5460
Epoch [15/40], Loss: 0.5384
Epoch [16/40], Loss: 0.5322
Epoch [17/40], Loss: 0.5272
Epoch [18/40], Loss: 0.5228
Epoch [19/40], Loss: 0.5184
Epoch [20/40], Loss: 0.5137
Epoch [21/40], Loss: 0.5085
Epoch [22/40], Loss: 0.5031
Epoch [23/40], Loss: 0.4974
Epoch [24/40], Loss: 0.4919
Epoch [25/40], Loss: 0.4866
Epoch [26/40], Loss: 0.4818
Epoch [27/40], Loss: 0.4774
Epoch [28/40], Loss: 0.4733
Epoch [29/40], Loss: 0.4693
Epoch [30/40], Loss: 0.4655
Epoch [31/40], Loss: 0.4616
Epoch [32/40], Loss: 0.4576
Epoch [33/40], Loss: 0.4535
Epoch [34/40], Loss: 0.4494
Epoch [35/40], Loss: 0.4453
Epoch [36/40], Loss: 0.4413
E

In [ ]:
train_y_tensor

tensor([0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
        0, 0, 1, 1])

In [ ]:


print(final_output.shape)  # Should be (999, 16, 152)
print(train_y_tensor.shape)  # Should be (999,)


torch.Size([100, 16, 96])
torch.Size([100])


In [ ]:
# Switch model to evaluation mode
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type
    final_output = final_output.float()

    # Perform inference
    outputs = model(final_output)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == train_y_tensor).sum().item() / train_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


Accuracy: 85.00%


In [ ]:
predicted

tensor([0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
        0, 0, 1, 1])

In [ ]:
train_y_tensor

tensor([0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
        1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
        0, 0, 1, 1])

In [ ]:
# Get the number of columns
cols = test2d.shape[1]

# Split the data
test_x = test2d[:, :cols-1]  # All columns except the last one
test_y = test2d[:, cols-1]   # The last column
test_y[np.where(test_y != 1)] = 0


In [ ]:
import torch

# Given parameters
order = 8
v_threshold = 1
bias = 0
gain = 1.0
num_neurons = 2 * order  # 16 neurons
timesteps = test_x.shape[1]  # Assuming there are 152 timesteps
#theta = 0.5
theta = 1.0
tau = 0.1
# Create an instance of EncoderLayer
encoder = EncoderLayer(order=order, num_neurons=num_neurons, v_threshold=v_threshold, gain=gain, bias=bias)
#train_x = test_x
#train_y = test_y
# Initialize a list to hold the stacked spikes for all samples
stacked_spikes = []
ldn_all_signals = []
# Process each sample in train_x
for i in range(test_x.shape[0]):  # Loop over all samples
    first_sample = test_x[i, :].reshape(1, -1)  # Reshape to match input format

    # Create an instance of PyTorchLDN for each sample
    ldn = PyTorchLDN(order, theta, tau)

    # Compute the LDN signals for the current sample
    ldn_first_sample_signals = ldn.get_ldn_sigs(first_sample)
    #print(ldn_first_sample_signals)

    # Initialize a list to hold spikes for the current sample
    current_sample_spikes = []
    ldn_all_signals.append(ldn_first_sample_signals)

    # Encoding over 152 timesteps using LDN output as input
    for t in range(timesteps):
        x_t = ldn_first_sample_signals[t]  # Extract the input for timestep `t` (size `order`)
        spikes = encoder.encode_layer(x_t)  # Process input and generate spikes
        current_sample_spikes.append(spikes)  # Collect spikes for current sample
        #print(spikes)
    # Stack the spikes for the current sample
    current_sample_spikes_tensor = torch.stack(current_sample_spikes)  # Shape will be [152, 16]
    if i==0:
      first_sample_xt = ldn_first_sample_signals
      first_sample_spikes = current_sample_spikes_tensor


    # Transpose to get shape [16, 152]
    combined_output = current_sample_spikes_tensor.transpose(0, 1)  # Shape will be [16, 152]

    # Append the combined output for the current sample
    stacked_spikes.append(combined_output)

# Stack all samples into a single tensor
final_output_test = torch.stack(stacked_spikes)  # Shape will be [6164, 16, 152]

# Print the final output shape
print("Final output shape:", final_output_test.shape)  # Should be [6164, 16, 152]


Final output shape: torch.Size([100, 16, 96])


In [ ]:


test_y_tensor = torch.tensor(test_y, dtype=torch.long)  # Ensure the correct dtype for classification

# Switch model to evaluation mode
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type
    final_output_test = final_output_test.float()

    # Perform inference
    outputs = model(final_output_test)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == test_y_tensor).sum().item() / test_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


RuntimeError: mat1 and mat2 shapes cannot be multiplied (100x1536 and 384x2)